# Manutenção Preditiva em Redutores Planetários por Análise de Vibração
## Análise multicondição com os datasets 1500 rpm / 10 Nm e 2700 rpm / 25 Nm


## 1. Configuração do Experimento Multicondição


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import kurtosis
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from IPython.display import display
import shap
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = PROJECT_ROOT / "images"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
ENV_FILE = PROJECT_ROOT / ".env"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"IMAGES_DIR: {IMAGES_DIR}")


## 2. Definição dos Datasets e Metadados


In [ ]:
fs = 10000
pontos_por_linha = 200
duracao_linha_s = pontos_por_linha / fs
pasta_dados = DATA_DIR

configuracoes_datasets = {
    "1500_10": {
        "arquivo_sufixo": "1500_10",
        "rotacao_entrada_rpm": 1500.0,
        "torque_nm": 10.0,
    },
    "2700_25": {
        "arquivo_sufixo": "2700_25",
        "rotacao_entrada_rpm": 2700.0,
        "torque_nm": 25.0,
    },
}

mapa_classes = {
    0: "Normal",
    1: "Desgaste Superficial",
    2: "Dente Trincado",
    3: "Dente Lascado",
    4: "Dente Ausente",
}

nomes_classes_ordenados = [mapa_classes[i] for i in sorted(mapa_classes)]
labels_classes_curtas = [f"Classe {i}" for i in sorted(mapa_classes)]

def rotulo_classe(classe):
    return f"{classe} - {mapa_classes.get(int(classe), 'Classe desconhecida')}"

for nome_dataset, config in configuracoes_datasets.items():
    for prefixo in ["x", "y", "z", "gt"]:
        caminho_arquivo = pasta_dados / f"{prefixo}_{config['arquivo_sufixo']}.npy"
        if not caminho_arquivo.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {caminho_arquivo}")

resumo_configuracoes = pd.DataFrame([
    {
        "dataset_operacao": nome_dataset,
        "arquivo_sufixo": config["arquivo_sufixo"],
        "rpm": config["rotacao_entrada_rpm"],
        "torque_nm": config["torque_nm"],
    }
    for nome_dataset, config in configuracoes_datasets.items()
])

display(resumo_configuracoes)


## 3. Carregamento dos Dados


In [ ]:
def carregar_dataset_operacao(nome_dataset, config):
    vibracao_eixo_x = np.load(pasta_dados / f"x_{config['arquivo_sufixo']}.npy")
    vibracao_eixo_y = np.load(pasta_dados / f"y_{config['arquivo_sufixo']}.npy")
    vibracao_eixo_z = np.load(pasta_dados / f"z_{config['arquivo_sufixo']}.npy")
    gt = np.load(pasta_dados / f"gt_{config['arquivo_sufixo']}.npy").astype(int)

    if not (len(vibracao_eixo_x) == len(vibracao_eixo_y) == len(vibracao_eixo_z) == len(gt)):
        raise ValueError(f"Inconsistência de tamanho no dataset {nome_dataset}.")

    df_vibracao_eixo_x = pd.DataFrame(vibracao_eixo_x)
    df_vibracao_eixo_y = pd.DataFrame(vibracao_eixo_y)
    df_vibracao_eixo_z = pd.DataFrame(vibracao_eixo_z)

    df_vibracao_eixo_x["classe"] = gt
    df_vibracao_eixo_y["classe"] = gt
    df_vibracao_eixo_z["classe"] = gt

    return {
        "dataset_operacao": nome_dataset,
        "arquivo_sufixo": config["arquivo_sufixo"],
        "rpm": float(config["rotacao_entrada_rpm"]),
        "torque_nm": float(config["torque_nm"]),
        "condicao_operacao": f"{int(config['rotacao_entrada_rpm'])} rpm / {int(config['torque_nm'])} Nm",
        "df_por_eixo": {
            "x": df_vibracao_eixo_x,
            "y": df_vibracao_eixo_y,
            "z": df_vibracao_eixo_z,
        },
        "gt": gt,
    }

datasets_brutos = {
    nome_dataset: carregar_dataset_operacao(nome_dataset, config)
    for nome_dataset, config in configuracoes_datasets.items()
}

resumo_datasets_brutos = []
for nome_dataset, info in datasets_brutos.items():
    resumo_datasets_brutos.append({
        "dataset_operacao": nome_dataset,
        "condicao_operacao": info["condicao_operacao"],
        "linhas": int(info["df_por_eixo"]["x"].shape[0]),
        "colunas_sinal": int(info["df_por_eixo"]["x"].shape[1] - 1),
        "classes_unicas": sorted(np.unique(info["gt"]).astype(int).tolist()),
    })

resumo_datasets_brutos = pd.DataFrame(resumo_datasets_brutos)
display(resumo_datasets_brutos)

## 4. Exportação de Amostras Brutas Contínuas para a Rock Pi

In [ ]:
duracao_exportacao_bruta_s = 5.0
linhas_exportacao_bruta = int(duracao_exportacao_bruta_s / duracao_linha_s)

pasta_saida_rockpi = OUTPUTS_DIR / "rockpi_test_samples"
pasta_saida_rockpi.mkdir(parents=True, exist_ok=True)

registros_exportacao_rockpi = []

for nome_dataset, info in datasets_brutos.items():
    for classe in sorted(mapa_classes):
        mascara_classe = info["gt"] == classe
        indices_origem = np.flatnonzero(mascara_classe)[:linhas_exportacao_bruta]

        if indices_origem.shape[0] < linhas_exportacao_bruta:
            raise ValueError(
                f"O dataset {nome_dataset} n?o possui linhas suficientes para exportar {duracao_exportacao_bruta_s:.1f} s da classe {classe}."
            )

        x_rows = info["df_por_eixo"]["x"].loc[mascara_classe].drop(columns="classe").iloc[:linhas_exportacao_bruta].to_numpy()
        y_rows = info["df_por_eixo"]["y"].loc[mascara_classe].drop(columns="classe").iloc[:linhas_exportacao_bruta].to_numpy()
        z_rows = info["df_por_eixo"]["z"].loc[mascara_classe].drop(columns="classe").iloc[:linhas_exportacao_bruta].to_numpy()
        gt_rows = info["gt"][indices_origem]

        caminho_npz = pasta_saida_rockpi / f"rockpi_raw_{nome_dataset}_classe_{classe}_{int(duracao_exportacao_bruta_s)}s.npz"

        np.savez(
            caminho_npz,
            dataset_operacao=nome_dataset,
            condicao_operacao=info["condicao_operacao"],
            rpm=info["rpm"],
            torque_nm=info["torque_nm"],
            classe=classe,
            classe_nome=mapa_classes[classe],
            fs=fs,
            pontos_por_linha=pontos_por_linha,
            duracao_linha_s=duracao_linha_s,
            duracao_total_s=duracao_exportacao_bruta_s,
            linha_inicial_classe_reconstruida=0,
            linha_final_classe_reconstruida=linhas_exportacao_bruta,
            indices_linhas_origem=indices_origem,
            x_rows=x_rows,
            y_rows=y_rows,
            z_rows=z_rows,
            gt_rows=gt_rows,
            x_flat=x_rows.reshape(-1),
            y_flat=y_rows.reshape(-1),
            z_flat=z_rows.reshape(-1),
        )

        registros_exportacao_rockpi.append({
            "dataset_operacao": nome_dataset,
            "condicao_operacao": info["condicao_operacao"],
            "classe": classe,
            "classe_nome": mapa_classes[classe],
            "duracao_s": duracao_exportacao_bruta_s,
            "linhas_exportadas": linhas_exportacao_bruta,
            "amostras_por_eixo": int(x_rows.size),
            "arquivo": caminho_npz.name,
        })

registros_exportacao_rockpi = pd.DataFrame(registros_exportacao_rockpi)
print(f"Exportação por classe: {duracao_exportacao_bruta_s:.1f} s por classe e código")
display(registros_exportacao_rockpi)


## 5. Segmentação do Sinal por Condição

In [ ]:
duracao_intervalo_s = 1.0
linhas_por_intervalo = int(duracao_intervalo_s / duracao_linha_s)
amostras_por_intervalo = linhas_por_intervalo * pontos_por_linha

def segmentar_dataset_por_condicao(dataset_info):
    segmentos_por_eixo_classe = {}

    for eixo, df_eixo in dataset_info["df_por_eixo"].items():
        segmentos_por_eixo_classe[eixo] = {}

        for classe in sorted(mapa_classes):
            matriz_classe = df_eixo[df_eixo["classe"] == classe].drop(columns="classe").to_numpy()
            if matriz_classe.size == 0:
                segmentos = np.empty((0, amostras_por_intervalo))
            else:
                if matriz_classe.size % amostras_por_intervalo != 0:
                    raise ValueError(
                        f"O total de amostras da classe {classe} no eixo {eixo} não é múltiplo de {amostras_por_intervalo}."
                    )
                segmentos = matriz_classe.reshape(-1, amostras_por_intervalo)

            segmentos_por_eixo_classe[eixo][classe] = segmentos

    return segmentos_por_eixo_classe

datasets_processados = {}
resumo_segmentacao = []

for nome_dataset, info in datasets_brutos.items():
    info_processado = dict(info)
    info_processado["segmentos_por_eixo_classe"] = segmentar_dataset_por_condicao(info)
    datasets_processados[nome_dataset] = info_processado

    for classe in sorted(mapa_classes):
        resumo_segmentacao.append({
            "dataset_operacao": nome_dataset,
            "condicao_operacao": info["condicao_operacao"],
            "classe": classe,
            "classe_nome": mapa_classes[classe],
            "segmentos_por_classe": int(info_processado["segmentos_por_eixo_classe"]["x"][classe].shape[0]),
        })

print(f"Duração do intervalo: {duracao_intervalo_s:.2f} s")
print(f"Linhas por intervalo: {linhas_por_intervalo}")
print(f"Amostras por intervalo: {amostras_por_intervalo}")

resumo_segmentacao = pd.DataFrame(resumo_segmentacao)
display(resumo_segmentacao)


## 6. Parâmetros Cinemáticos por Condição

In [ ]:
Zr1 = 100.0
Zs1 = 20.0
Zr2 = 100.0
Zs2 = 28.0
reducao_primeiro_estagio = 6.0

def calcular_parametros_cinematicos(rotacao_entrada_rpm):
    Fsh1 = rotacao_entrada_rpm / 60.0
    Fm1 = ((Zr1 * Zs1) / (Zr1 + Zs1)) * Fsh1
    Fh1 = (Zs1 / (Zr1 + Zs1)) * Fsh1
    Fcsd1 = Fm1 / Zs1
    Fcsl1 = 3 * Fcsd1

    Fsh2 = (rotacao_entrada_rpm / reducao_primeiro_estagio) / 60.0
    Fm2 = ((Zr2 * Zs2) / (Zr2 + Zs2)) * Fsh2
    Fh2 = (Zs2 / (Zr2 + Zs2)) * Fsh2
    Fcsd2 = Fm2 / Zs2
    Fcsl2 = 4 * Fcsd2

    return {
        "Fsh1": Fsh1,
        "Fm1": Fm1,
        "Fh1": Fh1,
        "Fcsd1": Fcsd1,
        "Fcsl1": Fcsl1,
        "Fsh2": Fsh2,
        "Fm2": Fm2,
        "Fh2": Fh2,
        "Fcsd2": Fcsd2,
        "Fcsl2": Fcsl2,
    }

resumo_cinematica = []
for nome_dataset, info in datasets_processados.items():
    parametros = calcular_parametros_cinematicos(info["rpm"])
    info["parametros_cinematicos"] = parametros

    resumo_cinematica.append({
        "dataset_operacao": nome_dataset,
        "condicao_operacao": info["condicao_operacao"],
        "Fm1_hz": parametros["Fm1"],
        "Fm2_hz": parametros["Fm2"],
        "Fh2_hz": parametros["Fh2"],
        "Fcsd2_hz": parametros["Fcsd2"],
        "Fcsl2_hz": parametros["Fcsl2"],
    })

resumo_cinematica = pd.DataFrame(resumo_cinematica)
display(resumo_cinematica)


## 7. Extração de Features por Condição

In [ ]:
largura_busca_fm_real_hz = 10.0
largura_banda_harmonica_hz = 10.0
ordens_harmonicas_fm2 = list(range(1, 6))
ordens_harmonicas_fm1 = list(range(1, 6))
janela_hann = np.hanning(amostras_por_intervalo)
frequencias_fft = np.fft.rfftfreq(amostras_por_intervalo, d=1 / fs)

def calcular_espectro_amplitude(sinal_segmento, janela):
    sinal_centrado = sinal_segmento - np.mean(sinal_segmento)
    sinal_janelado = sinal_centrado * janela
    espectro = np.fft.rfft(sinal_janelado)
    amplitude = np.abs(espectro) * 2.0 / janela.sum()
    amplitude[0] = 0.0
    return amplitude

def extrair_pico_na_banda(amplitude_espectral, frequencias, mascara_banda):
    if not np.any(mascara_banda):
        return np.nan, np.nan

    amplitudes_banda = amplitude_espectral[mascara_banda]
    frequencias_banda = frequencias[mascara_banda]
    indice_pico = int(np.argmax(amplitudes_banda))
    return float(amplitudes_banda[indice_pico]), float(frequencias_banda[indice_pico])

def extrair_soma_amplitudes_na_banda(amplitude_espectral, mascara_banda):
    if not np.any(mascara_banda):
        return np.nan
    return float(np.sum(np.square(amplitude_espectral[mascara_banda])))


def extrair_amplitude_maxima_na_banda(amplitude_espectral, mascara_banda):
    if not np.any(mascara_banda):
        return np.nan
    return float(np.max(amplitude_espectral[mascara_banda]))


def calcular_rms(sinal_segmento):
    return float(np.sqrt(np.mean(np.square(sinal_segmento))))


def calcular_kurtosis(sinal_segmento):
    return float(kurtosis(sinal_segmento, fisher=False, bias=False))


def calcular_peak_value(sinal_segmento):
    return float(np.max(np.abs(sinal_segmento)))


def calcular_crest_factor(sinal_segmento):
    rms = calcular_rms(sinal_segmento)
    if np.isclose(rms, 0.0):
        return np.nan
    return float(calcular_peak_value(sinal_segmento) / rms)


def extrair_features_condicao(nome_dataset, info):
    parametros = info["parametros_cinematicos"]
    segmentos_por_eixo_classe = info["segmentos_por_eixo_classe"]

    mascara_fm2_real = (
        (frequencias_fft >= parametros["Fm2"] - largura_busca_fm_real_hz)
        & (frequencias_fft <= parametros["Fm2"] + largura_busca_fm_real_hz)
    )
    mascara_fm1_real = (
        (frequencias_fft >= parametros["Fm1"] - largura_busca_fm_real_hz)
        & (frequencias_fft <= parametros["Fm1"] + largura_busca_fm_real_hz)
    )

    linhas_features = []

    for classe in sorted(mapa_classes):
        quantidade_segmentos = segmentos_por_eixo_classe["x"][classe].shape[0]

        for segmento_id in range(quantidade_segmentos):
            linha = {
                "dataset_operacao": nome_dataset,
                "condicao_operacao": info["condicao_operacao"],
                "rpm": info["rpm"],
                "torque_nm": info["torque_nm"],
                "classe": classe,
                "segmento_id": segmento_id,
                "tempo_inicial_s": segmento_id * duracao_intervalo_s,
                "tempo_final_s": (segmento_id + 1) * duracao_intervalo_s,
            }

            for eixo in ["x", "y", "z"]:
                sinal_segmento = segmentos_por_eixo_classe[eixo][classe][segmento_id]
                linha[f"rms_{eixo}"] = calcular_rms(sinal_segmento)
                linha[f"kurtosis_{eixo}"] = calcular_kurtosis(sinal_segmento)
                linha[f"peak_value_{eixo}"] = calcular_peak_value(sinal_segmento)
                linha[f"crest_factor_{eixo}"] = calcular_crest_factor(sinal_segmento)
                amplitude_espectral = calcular_espectro_amplitude(sinal_segmento, janela_hann)

                _, fm2_real = extrair_pico_na_banda(
                    amplitude_espectral,
                    frequencias_fft,
                    mascara_fm2_real,
                )
                linha[f"fm_real_{eixo}"] = fm2_real

                for ordem_harmonica in ordens_harmonicas_fm2:
                    frequencia_alvo = ordem_harmonica * fm2_real
                    mascara_harmonica = (
                        (frequencias_fft >= frequencia_alvo - largura_banda_harmonica_hz)
                        & (frequencias_fft <= frequencia_alvo + largura_banda_harmonica_hz)
                    )
                    linha[f"amp_fm_h{ordem_harmonica}_{eixo}"] = extrair_soma_amplitudes_na_banda(
                        amplitude_espectral,
                        mascara_harmonica,
                    )
                    linha[f"amp_max_fm_h{ordem_harmonica}_{eixo}"] = extrair_amplitude_maxima_na_banda(
                        amplitude_espectral,
                        mascara_harmonica,
                    )

                _, fm1_real = extrair_pico_na_banda(
                    amplitude_espectral,
                    frequencias_fft,
                    mascara_fm1_real,
                )
                linha[f"fm1_real_{eixo}"] = fm1_real

                for ordem_harmonica in ordens_harmonicas_fm1:
                    frequencia_alvo_fm1 = ordem_harmonica * fm1_real
                    mascara_harmonica_fm1 = (
                        (frequencias_fft >= frequencia_alvo_fm1 - largura_banda_harmonica_hz)
                        & (frequencias_fft <= frequencia_alvo_fm1 + largura_banda_harmonica_hz)
                    )
                    linha[f"amp_fm1_h{ordem_harmonica}_{eixo}"] = extrair_soma_amplitudes_na_banda(
                        amplitude_espectral,
                        mascara_harmonica_fm1,
                    )
                    linha[f"amp_max_fm1_h{ordem_harmonica}_{eixo}"] = extrair_amplitude_maxima_na_banda(
                        amplitude_espectral,
                        mascara_harmonica_fm1,
                    )

            linhas_features.append(linha)

    return pd.DataFrame(linhas_features)

features_por_dataset = {}
resumo_features = []

for nome_dataset, info in datasets_processados.items():
    features_condicao = extrair_features_condicao(nome_dataset, info)
    features_por_dataset[nome_dataset] = features_condicao
    resumo_features.append({
        "dataset_operacao": nome_dataset,
        "condicao_operacao": info["condicao_operacao"],
        "amostras": int(features_condicao.shape[0]),
        "colunas_totais": int(features_condicao.shape[1]),
    })

resumo_features = pd.DataFrame(resumo_features)
display(resumo_features)


## 8. Concatenação e Preparação para Modelagem

In [ ]:
features_modelo = pd.concat(features_por_dataset.values(), ignore_index=True)

colunas_identificacao = [
    "dataset_operacao",
    "condicao_operacao",
    "rpm",
    "torque_nm",
    "classe",
    "segmento_id",
    "tempo_inicial_s",
    "tempo_final_s",
]
colunas_features = [coluna for coluna in features_modelo.columns if coluna not in colunas_identificacao]

X_features = features_modelo[colunas_features].copy()
y_features = features_modelo["classe"].copy()

estrato_modelagem = (
    features_modelo["dataset_operacao"].astype(str)
    + "_classe_"
    + features_modelo["classe"].astype(str)
)

indices_modelagem, indices_teste = train_test_split(
    features_modelo.index.to_numpy(),
    test_size=0.3,
    random_state=42,
    stratify=estrato_modelagem,
)

estrato_treino_validacao = estrato_modelagem.loc[indices_modelagem]

indices_treino, indices_validacao = train_test_split(
    indices_modelagem,
    test_size=0.2,
    random_state=42,
    stratify=estrato_treino_validacao,
)

X_train = X_features.loc[indices_treino].copy()
X_val = X_features.loc[indices_validacao].copy()
X_test = X_features.loc[indices_teste].copy()
y_train = y_features.loc[indices_treino].copy()
y_val = y_features.loc[indices_validacao].copy()
y_test = y_features.loc[indices_teste].copy()

metadados_train = features_modelo.loc[indices_treino, colunas_identificacao].reset_index(drop=True)
metadados_val = features_modelo.loc[indices_validacao, colunas_identificacao].reset_index(drop=True)
metadados_test = features_modelo.loc[indices_teste, colunas_identificacao].reset_index(drop=True)

classes_roc = np.array(sorted(mapa_classes))
y_test_bin = label_binarize(y_test, classes=classes_roc)

resumo_split = pd.concat([
    metadados_train.assign(split="Treino"),
    metadados_val.assign(split="Validacao"),
    metadados_test.assign(split="Teste"),
], ignore_index=True)

resumo_split = (
    resumo_split.groupby(["split", "dataset_operacao", "classe"])
    .size()
    .reset_index(name="amostras")
    .sort_values(["split", "dataset_operacao", "classe"])
    .reset_index(drop=True)
)

print(f"Tamanho treino: {X_train.shape}")
print(f"Tamanho validacao: {X_val.shape}")
print(f"Tamanho teste: {X_test.shape}")
print(f"Tamanho total: {X_features.shape}")

display(resumo_split)
features_modelo.head()


## 9. Treinamento dos Modelos

In [ ]:
def calcular_metricas_multiclasse(nome_modelo, y_true, y_pred):
    return {
        "modelo": nome_modelo,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

random_forest = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,              # ADICIONADO: Impede ?rvores infinitamente profundas
    min_samples_split=10,     # ADICIONADO: Exige 10 amostras para tentar dividir um n?
    min_samples_leaf=10,      # ADICIONADO: Exige 10 amostras para formar uma folha final
    max_features="sqrt",     # Padr?o, mas bom garantir (ajuda a descorrelacionar ?rvores)
    random_state=42,
    n_jobs=-1,                # Usa todos os n?cleos dispon?veis
)

xgboost_model = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.6,
    colsample_bytree=0.6,
    min_child_weight=10,
    reg_alpha=10.0,
    reg_lambda=10.0,
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    random_state=42,
    verbosity=0,
    early_stopping_rounds=25,
)

lightgbm_model = LGBMClassifier(
    objective="multiclass",
    num_class=5,
    n_estimators=500,          # Aumentado para trabalhar com Early Stopping
    learning_rate=0.01,        # Reduzido para passos menores
    max_depth=4,              # Reduzido de 6 para 4
    num_leaves=15,            # Deve ser menor que 2^(max_depth)
    min_child_samples=20,     # Exige 20 amostras por folha
    subsample=0.5,            # Reduzido para aumentar aleatoriedade
    subsample_freq=1,         # Ativa o subsample a cada itera??o
    colsample_bytree=0.5,     # Reduzido para parear com o XGBoost
    reg_alpha=10.0,
    reg_lambda=10.0,
    random_state=42,
    verbosity=-1,
)

modelos = {
    "RandomForest": random_forest,
    "XGBoost": xgboost_model,
    "LightGBM": lightgbm_model,
}

resultados_modelos = {}
metricas_globais = []

for nome_modelo, modelo in modelos.items():
    if nome_modelo == "XGBoost":
        modelo.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
    elif nome_modelo == "LightGBM":
        modelo.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="multi_logloss",
            callbacks=[early_stopping(stopping_rounds=25, verbose=False)],
        )
    else:
        modelo.fit(X_train, y_train)

    y_pred = np.asarray(modelo.predict(X_test)).astype(int).ravel()
    y_proba = modelo.predict_proba(X_test)

    resultados_modelos[nome_modelo] = {
        "modelo": modelo,
        "y_pred": y_pred,
        "y_proba": y_proba,
    }
    metricas_globais.append(calcular_metricas_multiclasse(nome_modelo, y_test, y_pred))

comparacao_modelos = pd.DataFrame(metricas_globais).sort_values(
    ["f1_macro", "accuracy"],
    ascending=False,
).reset_index(drop=True)

y_pred_rf = resultados_modelos["RandomForest"]["y_pred"]
y_proba_rf = resultados_modelos["RandomForest"]["y_proba"]
y_pred_xgb = resultados_modelos["XGBoost"]["y_pred"]
y_proba_xgb = resultados_modelos["XGBoost"]["y_proba"]
y_pred_lgbm = resultados_modelos["LightGBM"]["y_pred"]
y_proba_lgbm = resultados_modelos["LightGBM"]["y_proba"]

display(comparacao_modelos)


## 10. Avaliação Global e por Condição Operacional

In [ ]:
def calcular_metricas_por_condicao(nome_modelo, y_true, y_pred, metadados):
    metricas = []

    for (dataset_operacao, condicao_operacao), grupo in metadados.groupby(["dataset_operacao", "condicao_operacao"]):
        mascara = metadados.index.isin(grupo.index)
        linha = calcular_metricas_multiclasse(nome_modelo, y_true[mascara], y_pred[mascara])
        linha["dataset_operacao"] = dataset_operacao
        linha["condicao_operacao"] = condicao_operacao
        linha["amostras_teste"] = int(mascara.sum())
        metricas.append(linha)

    return pd.DataFrame(metricas)

comparacao_modelos_por_condicao = pd.concat([
    calcular_metricas_por_condicao("RandomForest", y_test, y_pred_rf, metadados_test),
    calcular_metricas_por_condicao("XGBoost", y_test, y_pred_xgb, metadados_test),
    calcular_metricas_por_condicao("LightGBM", y_test, y_pred_lgbm, metadados_test),
], ignore_index=True)

display(comparacao_modelos)
display(comparacao_modelos_por_condicao)


In [ ]:
def plotar_avaliacao_modelo(nome_modelo, y_true, y_pred, y_proba, sufixo_arquivo):
    print(f"Métricas - {nome_modelo}")
    metricas_modelo = comparacao_modelos[comparacao_modelos["modelo"] == nome_modelo]
    print(metricas_modelo.to_string(index=False))
    print(classification_report(
        y_true,
        y_pred,
        labels=sorted(mapa_classes),
        target_names=nomes_classes_ordenados,
        zero_division=0,
    ))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        ax=axes[0],
        cmap="Blues",
        colorbar=False,
        display_labels=labels_classes_curtas,
    )
    axes[0].set_title(f"Matriz de confusão - {nome_modelo}")

    for indice_classe, classe in enumerate(classes_roc):
        fpr_classe, tpr_classe, _ = roc_curve(y_test_bin[:, indice_classe], y_proba[:, indice_classe])
        auc_classe = auc(fpr_classe, tpr_classe)
        axes[1].plot(
            fpr_classe,
            tpr_classe,
            linewidth=1.5,
            label=f"Classe {classe} (AUC = {auc_classe:.3f})",
        )

    axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[1].set_title(f"Curva ROC - {nome_modelo}")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    pasta_imagens = IMAGES_DIR
    pasta_imagens.mkdir(exist_ok=True)
    caminho_arquivo = pasta_imagens / f"avaliacao_{sufixo_arquivo}_multicondicao.png"
    fig.savefig(caminho_arquivo, dpi=300, bbox_inches="tight")
    print(f"Gráfico salvo em: {caminho_arquivo}")
    plt.show()


In [ ]:
plotar_avaliacao_modelo("RandomForest", y_test, y_pred_rf, y_proba_rf, "randomforest")


In [ ]:
plotar_avaliacao_modelo("XGBoost", y_test, y_pred_xgb, y_proba_xgb, "xgboost")


In [ ]:
plotar_avaliacao_modelo("LightGBM", y_test, y_pred_lgbm, y_proba_lgbm, "lightgbm")


## 11. Explicabilidade com SHAP

In [ ]:
amostras_explicacao_por_grupo = 1
max_amostras_shap_global = 200

metadados_test_explicacao = metadados_test.copy()
metadados_test_explicacao["indice_teste_pos"] = np.arange(metadados_test_explicacao.shape[0])
metadados_test_explicacao["classe_real_nome"] = metadados_test_explicacao["classe"].map(mapa_classes)

grupos_explicacao = []
for _, df_grupo in metadados_test_explicacao.groupby(["dataset_operacao", "classe"], sort=True):
    grupos_explicacao.append(
        df_grupo.sample(n=min(amostras_explicacao_por_grupo, len(df_grupo)), random_state=42)
    )

amostras_explicacao_info = (
    pd.concat(grupos_explicacao, ignore_index=True)
    .sort_values(["dataset_operacao", "classe", "indice_teste_pos"])
    .reset_index(drop=True)
)

indices_explicacao = amostras_explicacao_info["indice_teste_pos"].to_numpy()
X_explicacao = X_test.reset_index(drop=True).iloc[indices_explicacao].copy()
y_explicacao = y_test.reset_index(drop=True).iloc[indices_explicacao].copy()
X_shap_global = (
    X_test.reset_index(drop=True)
    .sample(n=min(max_amostras_shap_global, X_test.shape[0]), random_state=42)
    .reset_index(drop=True)
)

resumo_amostras_explicacao = amostras_explicacao_info[
    ["dataset_operacao", "condicao_operacao", "classe", "classe_real_nome", "segmento_id", "tempo_inicial_s", "tempo_final_s"]
].copy()

print(f"Amostras selecionadas para explicação local: {X_explicacao.shape[0]}")
print(f"Amostras usadas para SHAP global: {X_shap_global.shape[0]}")
display(resumo_amostras_explicacao)

In [ ]:
def organizar_shap_multiclasse(shap_values, n_classes):
    if hasattr(shap_values, "values"):
        shap_array = shap_values.values
    else:
        shap_array = shap_values

    if isinstance(shap_array, list):
        return np.stack(shap_array, axis=0)

    shap_array = np.asarray(shap_array)

    if shap_array.ndim == 3 and shap_array.shape[2] == n_classes:
        return np.moveaxis(shap_array, 2, 0)

    if shap_array.ndim == 3 and shap_array.shape[0] == n_classes:
        return shap_array

    raise ValueError(f"Formato de SHAP não suportado: {shap_array.shape}")


def calcular_shap_multiclasse(modelo, X_ref):
    explainer = shap.TreeExplainer(modelo)
    shap_raw = explainer.shap_values(X_ref)
    return organizar_shap_multiclasse(shap_raw, n_classes=len(classes_roc))


def calcular_importancia_global_shap(shap_classes, colunas_features, nome_modelo, top_k=15):
    impacto_medio = np.mean(np.abs(shap_classes), axis=(0, 1))
    tabela = pd.DataFrame({
        "modelo": nome_modelo,
        "feature": list(colunas_features),
        "impacto_medio_absoluto": impacto_medio,
    }).sort_values("impacto_medio_absoluto", ascending=False).reset_index(drop=True)
    tabela["rank"] = np.arange(1, tabela.shape[0] + 1)
    return tabela.head(top_k).copy()


def tabela_shap_local(shap_classes, X_ref, info_ref, y_proba_ref, nome_modelo, top_k=10):
    classes_preditas = np.argmax(y_proba_ref, axis=1)
    probabilidades_preditas = np.max(y_proba_ref, axis=1)
    registros = []

    for i in range(X_ref.shape[0]):
        classe_predita = int(classes_preditas[i])
        valores_shap = shap_classes[classe_predita, i, :]
        ordem = np.argsort(np.abs(valores_shap))[::-1][:top_k]

        for rank, indice_feature in enumerate(ordem, start=1):
            registros.append({
                "modelo": nome_modelo,
                "amostra_explicacao": i,
                "dataset_operacao": info_ref.loc[i, "dataset_operacao"],
                "condicao_operacao": info_ref.loc[i, "condicao_operacao"],
                "classe_real": int(info_ref.loc[i, "classe"]),
                "classe_real_nome": mapa_classes[int(info_ref.loc[i, "classe"])],
                "classe_predita": classe_predita,
                "classe_predita_nome": mapa_classes[classe_predita],
                "probabilidade_predita": float(probabilidades_preditas[i]),
                "segmento_id": int(info_ref.loc[i, "segmento_id"]),
                "tempo_inicial_s": float(info_ref.loc[i, "tempo_inicial_s"]),
                "tempo_final_s": float(info_ref.loc[i, "tempo_final_s"]),
                "rank": rank,
                "feature": X_ref.columns[indice_feature],
                "valor_feature": float(X_ref.iloc[i, indice_feature]),
                "shap_value": float(valores_shap[indice_feature]),
                "impacto_absoluto": float(abs(valores_shap[indice_feature])),
            })

    return pd.DataFrame(registros)


def plotar_importancia_global_shap(tabela_global, nome_modelo):
    top_plot = tabela_global.sort_values("impacto_medio_absoluto", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top_plot["feature"], top_plot["impacto_medio_absoluto"], color="tab:blue")
    ax.set_title(f"SHAP global - {nome_modelo}")
    ax.set_xlabel("Mean absolute SHAP value")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    pasta_imagens = IMAGES_DIR
    pasta_imagens.mkdir(exist_ok=True)
    caminho = pasta_imagens / f"shap_global_{nome_modelo.lower()}_multicondicao.png"
    fig.savefig(caminho, dpi=300, bbox_inches="tight")
    print(f"Gráfico salvo em: {caminho}")
    plt.show()


def plotar_shap_local(tabela_local, nome_modelo):
    quantidade_amostras = tabela_local["amostra_explicacao"].nunique()
    fig, axes = plt.subplots(quantidade_amostras, 1, figsize=(14, 3 * quantidade_amostras), sharex=False)
    if quantidade_amostras == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        tabela_amostra = tabela_local[tabela_local["amostra_explicacao"] == i].sort_values("impacto_absoluto", ascending=True)
        cores = ["tab:blue" if valor >= 0 else "tab:orange" for valor in tabela_amostra["shap_value"]]
        ax.barh(tabela_amostra["feature"], tabela_amostra["shap_value"], color=cores)
        linha_info = tabela_amostra.iloc[-1]
        ax.set_title(
            f"Amostra {i} | {linha_info['dataset_operacao']} | Real: {linha_info['classe_real_nome']} | "
            f"Predita: {linha_info['classe_predita_nome']} ({linha_info['probabilidade_predita']:.3f})"
        )
        ax.set_xlabel("Valor SHAP")
        ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()

    pasta_imagens = IMAGES_DIR
    pasta_imagens.mkdir(exist_ok=True)
    caminho = pasta_imagens / f"shap_local_{nome_modelo.lower()}_multicondicao.png"
    fig.savefig(caminho, dpi=300, bbox_inches="tight")
    print(f"Gráfico salvo em: {caminho}")
    plt.show()

In [ ]:
resultados_shap = {}
tabelas_globais_shap = []
tabelas_locais_shap = []

for nome_modelo, resultado in resultados_modelos.items():
    modelo = resultado["modelo"]
    y_proba_explicacao = modelo.predict_proba(X_explicacao)

    shap_classes_global = calcular_shap_multiclasse(modelo, X_shap_global)
    tabela_global = calcular_importancia_global_shap(
        shap_classes_global,
        X_shap_global.columns,
        nome_modelo,
        top_k=15,
    )

    shap_classes_local = calcular_shap_multiclasse(modelo, X_explicacao)
    tabela_local = tabela_shap_local(
        shap_classes_local,
        X_explicacao,
        amostras_explicacao_info.reset_index(drop=True),
        y_proba_explicacao,
        nome_modelo,
        top_k=10,
    )

    resultados_shap[nome_modelo] = {
        "global": tabela_global,
        "local": tabela_local,
    }
    tabelas_globais_shap.append(tabela_global)
    tabelas_locais_shap.append(tabela_local)

    print(f"Top features SHAP globais - {nome_modelo}")
    display(tabela_global)
    plotar_importancia_global_shap(tabela_global, nome_modelo)

    print(f"Top features SHAP locais - {nome_modelo}")
    display(tabela_local)
    plotar_shap_local(tabela_local, nome_modelo)

comparacao_shap_global = pd.concat(tabelas_globais_shap, ignore_index=True)
comparacao_shap_local = pd.concat(tabelas_locais_shap, ignore_index=True)

In [ ]:
display(comparacao_shap_global)
display(comparacao_shap_local)

## 12. Geração de Prompt para LLM Local

In [ ]:
import re


def formatar_valor_feature(valor_feature):
    valor = float(valor_feature)
    if np.isnan(valor):
        return "nan"
    return f"{valor:.6f}"


def descrever_feature_prompt_local(feature, valor_feature, shap_value):
    contribuicao = "contribuiu positivamente para a classe predita" if shap_value >= 0 else "contribuiu negativamente para a classe predita"
    valor_txt = formatar_valor_feature(valor_feature)

    match_amp_max_fm2 = re.match(r"amp_max_fm_h(\d+)_([xyz])", feature)
    if match_amp_max_fm2:
        ordem, eixo = match_amp_max_fm2.groups()
        return f"Eixo {eixo.upper()}: amplitude máxima observada = {valor_txt} na banda da {ordem}ª harmônica da frequência de engrenamento do 2º estágio; essa variável {contribuicao}."

    match_energy_fm2 = re.match(r"amp_fm_h(\d+)_([xyz])", feature)
    if match_energy_fm2:
        ordem, eixo = match_energy_fm2.groups()
        return f"Eixo {eixo.upper()}: energia observada = {valor_txt} na banda da {ordem}ª harmônica da frequência de engrenamento do 2º estágio; essa variável {contribuicao}."

    match_amp_max_fm1 = re.match(r"amp_max_fm1_h(\d+)_([xyz])", feature)
    if match_amp_max_fm1:
        ordem, eixo = match_amp_max_fm1.groups()
        return f"Eixo {eixo.upper()}: amplitude máxima observada = {valor_txt} na banda da {ordem}ª harmônica da frequência de engrenamento do 1º estágio; essa variável {contribuicao}."

    match_energy_fm1 = re.match(r"amp_fm1_h(\d+)_([xyz])", feature)
    if match_energy_fm1:
        ordem, eixo = match_energy_fm1.groups()
        return f"Eixo {eixo.upper()}: energia observada = {valor_txt} na banda da {ordem}ª harmônica da frequência de engrenamento do 1º estágio; essa variável {contribuicao}."

    match_rms = re.match(r"rms_([xyz])", feature)
    if match_rms:
        eixo = match_rms.group(1)
        return f"Eixo {eixo.upper()}: RMS observado = {valor_txt}; essa variável {contribuicao}."

    match_kurtosis = re.match(r"kurtosis_([xyz])", feature)
    if match_kurtosis:
        eixo = match_kurtosis.group(1)
        return f"Eixo {eixo.upper()}: curtose observada = {valor_txt}; essa vari?vel {contribuicao}."

    match_peak_value = re.match(r"peak_value_([xyz])", feature)
    if match_peak_value:
        eixo = match_peak_value.group(1)
        return f"Eixo {eixo.upper()}: valor de pico observado = {valor_txt}; essa vari?vel {contribuicao}."

    match_crest_factor = re.match(r"crest_factor_([xyz])", feature)
    if match_crest_factor:
        eixo = match_crest_factor.group(1)
        return f"Eixo {eixo.upper()}: fator de crista observado = {valor_txt}; essa vari?vel {contribuicao}."

    match_fm2 = re.match(r"fm_real_([xyz])", feature)
    if match_fm2:
        eixo = match_fm2.group(1)
        return f"Eixo {eixo.upper()}: frequência real de engrenamento do 2º estágio observada = {valor_txt} Hz; essa variável {contribuicao}."

    match_fm1 = re.match(r"fm1_real_([xyz])", feature)
    if match_fm1:
        eixo = match_fm1.group(1)
        return f"Eixo {eixo.upper()}: frequência real de engrenamento do 1º estágio observada = {valor_txt} Hz; essa variável {contribuicao}."

    return f"{feature}: valor observado = {valor_txt}; essa variável {contribuicao}."


def obter_tabela_shap_fonte(nome_modelo):
    if nome_modelo not in resultados_shap:
        raise ValueError(f"Fonte SHAP inválida: {nome_modelo}")
    return resultados_shap[nome_modelo]["local"].copy()


def montar_evidencias_llm(tabela_local, amostra_explicacao, max_evidencias=5):
    tabela_amostra = (
        tabela_local[tabela_local["amostra_explicacao"] == amostra_explicacao]
        .sort_values("impacto_absoluto", ascending=False)
        .head(max_evidencias)
        .copy()
    )
    evidencias = []
    for i, row in enumerate(tabela_amostra.itertuples(index=False), start=1):
        evidencias.append(f"{i}. {descrever_feature_prompt_local(row.feature, row.valor_feature, row.shap_value)}")
    return tabela_amostra, "\n".join(evidencias)


system_prompt_qwen_local = """Você é um especialista em análise de vibração aplicada à manutenção preditiva de redutores planetários.

Tarefa:
Gerar uma explicação técnica curta para uma classificação de falha com base somente nas evidências fornecidas.

Regras:
- Use apenas as informações recebidas.
- Não invente frequências, componentes, causas ou sintomas não citados.
- Não trate uma variável como alta, baixa, elevada, reduzida ou anormal sem referência explícita no prompt.
- Interprete as evidências como variáveis que contribuíram para a classe predita, usando os valores observados e o contexto mecânico disponível.
- Não mencione SHAP, modelo, IA, prompt ou explicabilidade.
- Não forneça ações recomendadas.
- Seja técnico, direto e conciso.
- Se a evidência for moderada, use linguagem cautelosa, como "compatível com" ou "sugere".
- Responda exatamente com estas duas seções:
Interpretação Vibracional:
Interpretação Mecânica:"""

In [ ]:
fonte_shap_llm_local = "LightGBM"  # RandomForest, XGBoost ou LightGBM
amostra_explicacao_llm_local = 0
max_evidencias_llm_local = 5

tabela_shap_fonte_llm_local = obter_tabela_shap_fonte(fonte_shap_llm_local)
tabela_llm_amostra, evidencias_traduzidas_llm = montar_evidencias_llm(
    tabela_shap_fonte_llm_local,
    amostra_explicacao_llm_local,
    max_evidencias=max_evidencias_llm_local,
)

if tabela_llm_amostra.empty:
    raise ValueError("Nenhuma evidência SHAP encontrada para a amostra selecionada.")

linha_ref_llm = tabela_llm_amostra.iloc[0]

user_prompt_qwen_local = f"""Equipamento: redutor planetário de 2 estágios.
Componente monitorado: engrenagem solar do 2º estágio.
Condição operacional: {linha_ref_llm['condicao_operacao']}.

Classes possíveis:
0 = Normal
1 = Desgaste Superficial
2 = Dente Trincado
3 = Dente Lascado
4 = Dente Ausente

Legenda física das variáveis:
- RMS: nível global de vibração do segmento.
- Amplitude máxima na banda: maior amplitude espectral dentro da banda analisada.
- Energia na banda: soma da amplitude ao quadrado dentro da banda analisada.
- Fm: frequência de engrenamento.
- Harmônica de Fm: múltiplo inteiro da frequência de engrenamento.
- Banda de ressonância: faixa de alta frequência associada à excitação estrutural.
- Modulação lateral: variação espectral ao redor da frequência de engrenamento, compatível com defeitos distribuídos ou localizados.

Classificação predita: {linha_ref_llm['classe_predita_nome']}.
Probabilidade da classe predita: {linha_ref_llm['probabilidade_predita']:.3f}.

Evidências principais:
{evidencias_traduzidas_llm}

Escreva a resposta final."""

prompt_llm_local = {
    "fonte_shap": fonte_shap_llm_local,
    "amostra_explicacao": int(amostra_explicacao_llm_local),
    "system_prompt": system_prompt_qwen_local,
    "user_prompt": user_prompt_qwen_local,
}

display(tabela_llm_amostra[[
    "dataset_operacao",
    "classe_real_nome",
    "classe_predita_nome",
    "probabilidade_predita",
    "feature",
    "shap_value",
    "impacto_absoluto",
]])

print("SYSTEM PROMPT:\n")
print(system_prompt_qwen_local)
print("\nUSER PROMPT:\n")
print(user_prompt_qwen_local)